# Breast Cancer Classification Using Vision Transformers
## Complete Training Notebook

This notebook contains the complete training pipeline for breast cancer classification using:
- **MaxViT** - Multi-Axis Vision Transformer
- **MobileViT** - Lightweight Mobile Vision Transformer
- **EfficientViT** - Optimized Efficient Vision Transformer
- **HybridViT** - Novel Feature-Level Fusion of MaxViT + MobileViT

**Datasets:** DDSM (Mammography), BreakHis (Histopathology), BUS_UC (Ultrasound)

---
## 1. Install Dependencies

In [ ]:
# Uncomment and run if packages are not installed
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
# !pip install timm
# !pip install scikit-learn
# !pip install matplotlib seaborn
# !pip install tqdm
# !pip install pillow

---
## 2. Import Libraries

In [ ]:
import os
import re
import time
import copy
import json
import math
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, average_precision_score,
    matthews_corrcoef, cohen_kappa_score
)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

---
## 3. Configuration

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

# Paths - UPDATE THESE TO YOUR DATASET LOCATION
BASE_DIR = os.path.dirname(os.path.abspath("__file__"))
DATASET_DIR = os.path.join(BASE_DIR, "Dataset Zip")

# Individual dataset paths
DDSM_BENIGN = os.path.join(DATASET_DIR, "DDSM Dataset", "DDSM Dataset", "Benign Masses")
DDSM_MALIGNANT = os.path.join(DATASET_DIR, "DDSM Dataset", "DDSM Dataset", "Malignant Masses")

BUS_UC_BENIGN = os.path.join(DATASET_DIR, "BUS_UC", "BUS_UC", "Benign", "images")
BUS_UC_MALIGNANT = os.path.join(DATASET_DIR, "BUS_UC", "BUS_UC", "Malignant", "images")

BREAKHIS_BASE = os.path.join(DATASET_DIR, "dataset_cancer_v1", "dataset_cancer_v1", "classificacao_binaria")
BREAKHIS_MAGNIFICATIONS = ["40X", "100X", "200X", "400X"]

# Output directories
RESULTS_DIR = os.path.join(BASE_DIR, "results")
CHECKPOINT_DIR = os.path.join(BASE_DIR, "checkpoints")

# Data Configuration
IMAGE_SIZE = 224
NUM_CLASSES = 2
CLASS_NAMES = ["Benign", "Malignant"]

# Train / Validation / Test split ratios
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
RANDOM_SEED = 42

# Authenticity / Evaluation Strictness
AUTHENTIC_EVAL = True
DDSM_EXCLUDE_AUGMENTED_VARIANTS = True
DDSM_EXCLUDE_AMBIGUOUS_CASES = True

# Training Hyperparameters
BATCH_SIZE = 16
NUM_EPOCHS = 15
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-2
NUM_WORKERS = 4
PIN_MEMORY = True

# Learning rate schedule
WARMUP_EPOCHS = 1
LR_MIN = 1e-6

# Regularization
LABEL_SMOOTHING = 0.1
DROPOUT_RATE = 0.3
GRAD_CLIP_MAX_NORM = 1.0
MIXUP_ALPHA = 0.2

# Progressive unfreezing
FREEZE_BACKBONE_EPOCHS = 1

# Early stopping
EARLY_STOPPING_PATIENCE = 7
EARLY_STOPPING_MIN_DELTA = 1e-4

# Data Augmentation
AUGMENTATION = {
    "horizontal_flip": True,
    "vertical_flip": True,
    "random_rotation": 15,
    "color_jitter_brightness": 0.2,
    "color_jitter_contrast": 0.2,
    "color_jitter_saturation": 0.1,
    "color_jitter_hue": 0.05,
    "random_affine_degrees": 10,
    "random_affine_translate": (0.1, 0.1),
    "random_erasing_prob": 0.15,
}

# Model configurations (timm model names)
MODELS = {
    "maxvit": {
        "name": "maxvit_tiny_tf_224.in1k",
        "pretrained": True,
        "description": "MaxViT - CNN + Transformer hybrid with multi-axis attention",
    },
    "mobilevit": {
        "name": "mobilevitv2_100.cvnets_in1k",
        "pretrained": True,
        "description": "MobileViT - Lightweight hybrid for efficiency",
    },
    "efficientvit": {
        "name": "efficientvit_b1.r224_in1k",
        "pretrained": True,
        "description": "EfficientViT - Optimized for fast inference",
    },
}

# Create output directories
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("Configuration loaded!")
print(f"Dataset directory: {DATASET_DIR}")
print(f"Results directory: {RESULTS_DIR}")

---
## 4. Dataset Loading Functions

In [ ]:
# ============================================================
# DATASET UTILITIES
# ============================================================

def _ddsm_case_id(filename):
    """Return the DDSM case identifier from a filename."""
    return filename.split('.')[0]


def _is_ddsm_augmented_variant(filename):
    """Detect DDSM generated variants like '... (2).png'."""
    stem = os.path.splitext(filename)[0]
    return bool(re.search(r"\(\d+\)$", stem))


def _extract_patient_id(filename, source_name):
    """
    Extract a patient/slide-level group ID from a filename.
    All images belonging to the same patient/slide will share the same group ID.
    """
    if source_name.startswith("BreakHis"):
        parts = filename.split('-')
        if len(parts) >= 3:
            return f"breakhis_{parts[1]}-{parts[2]}"
        return f"breakhis_{filename}"
    elif source_name == "DDSM":
        case_id = _ddsm_case_id(filename)
        return f"ddsm_{case_id}"
    elif source_name == "BUS_UC":
        return f"busuc_{filename}"
    return f"unknown_{filename}"


def _collect_ddsm_entries(image_exts):
    """Collect DDSM image records with optional strict curation."""
    entries = []
    removed_augmented = 0

    def scan_directory(directory, label):
        nonlocal removed_augmented
        if not os.path.isdir(directory):
            print(f"  [WARNING] Directory not found: {directory}")
            return

        for fname in os.listdir(directory):
            if os.path.splitext(fname)[1].lower() not in image_exts:
                continue

            if AUTHENTIC_EVAL and DDSM_EXCLUDE_AUGMENTED_VARIANTS and _is_ddsm_augmented_variant(fname):
                removed_augmented += 1
                continue

            case_id = _ddsm_case_id(fname)
            entries.append(
                (os.path.join(directory, fname), label, "DDSM", f"ddsm_{case_id}", case_id)
            )

    scan_directory(DDSM_BENIGN, 0)
    scan_directory(DDSM_MALIGNANT, 1)

    removed_ambiguous = 0
    if AUTHENTIC_EVAL and DDSM_EXCLUDE_AMBIGUOUS_CASES:
        case_to_labels = {}
        for _, label, _, _, case_id in entries:
            case_to_labels.setdefault(case_id, set()).add(label)

        ambiguous_cases = {case_id for case_id, lbls in case_to_labels.items() if len(lbls) > 1}
        if ambiguous_cases:
            filtered = [record for record in entries if record[4] not in ambiguous_cases]
            removed_ambiguous = len(entries) - len(filtered)
            entries = filtered

    benign_count = sum(1 for _, label, _, _, _ in entries if label == 0)
    malignant_count = sum(1 for _, label, _, _, _ in entries if label == 1)

    stats = {
        "benign": benign_count,
        "malignant": malignant_count,
        "removed_augmented": removed_augmented,
        "removed_ambiguous": removed_ambiguous,
    }
    return entries, stats


def collect_all_image_paths():
    """Collect all image paths from the three datasets with their labels."""
    image_paths = []
    labels = []
    sources = []
    group_ids = []
    image_exts = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

    def add_images(directory, label, source_name):
        if not os.path.isdir(directory):
            print(f"  [WARNING] Directory not found: {directory}")
            return 0
        count = 0
        for f in os.listdir(directory):
            if os.path.splitext(f)[1].lower() in image_exts:
                image_paths.append(os.path.join(directory, f))
                labels.append(label)
                sources.append(source_name)
                group_ids.append(_extract_patient_id(f, source_name))
                count += 1
        return count

    print("=" * 60)
    print("Collecting images from all datasets...")
    print("=" * 60)

    # 1. DDSM Dataset
    ddsm_entries, ddsm_stats = _collect_ddsm_entries(image_exts)
    for img_path, label, source, group_id, _case_id in ddsm_entries:
        image_paths.append(img_path)
        labels.append(label)
        sources.append(source)
        group_ids.append(group_id)

    print(f"  DDSM Benign:     {ddsm_stats['benign']} images")
    print(f"  DDSM Malignant:  {ddsm_stats['malignant']} images")

    # 2. BUS_UC Dataset
    n = add_images(BUS_UC_BENIGN, 0, "BUS_UC")
    print(f"  BUS_UC Benign:   {n} images")
    n = add_images(BUS_UC_MALIGNANT, 1, "BUS_UC")
    print(f"  BUS_UC Malignant:{n} images")

    # 3. BreakHis Dataset (all magnifications)
    for mag in BREAKHIS_MAGNIFICATIONS:
        benign_dir = os.path.join(BREAKHIS_BASE, mag, "benign")
        malignant_dir = os.path.join(BREAKHIS_BASE, mag, "malignant")
        nb = add_images(benign_dir, 0, f"BreakHis_{mag}")
        nm = add_images(malignant_dir, 1, f"BreakHis_{mag}")
        print(f"  BreakHis {mag}: benign={nb}, malignant={nm}")

    total = len(image_paths)
    n_benign = labels.count(0)
    n_malignant = labels.count(1)
    n_groups = len(set(group_ids))
    print("-" * 60)
    print(f"  TOTAL: {total} images  (Benign: {n_benign}, Malignant: {n_malignant})")
    print(f"  Unique patient/slide groups: {n_groups}")
    print("=" * 60)

    return image_paths, labels, sources, group_ids


print("Dataset utilities loaded!")

---
## 5. Data Transforms and Dataset Class

In [ ]:
# ============================================================
# TRANSFORMS AND DATASET CLASS
# ============================================================

def get_transforms(is_training=True):
    """Get data transforms for training or evaluation."""
    if is_training:
        aug = AUGMENTATION
        transform = transforms.Compose([
            transforms.Resize((IMAGE_SIZE + 32, IMAGE_SIZE + 32)),
            transforms.RandomCrop(IMAGE_SIZE),
            transforms.RandomHorizontalFlip() if aug["horizontal_flip"] else transforms.Lambda(lambda x: x),
            transforms.RandomVerticalFlip() if aug["vertical_flip"] else transforms.Lambda(lambda x: x),
            transforms.RandomRotation(aug["random_rotation"]),
            transforms.ColorJitter(
                brightness=aug["color_jitter_brightness"],
                contrast=aug["color_jitter_contrast"],
                saturation=aug["color_jitter_saturation"],
                hue=aug["color_jitter_hue"],
            ),
            transforms.RandomAffine(
                degrees=aug["random_affine_degrees"],
                translate=aug["random_affine_translate"],
            ),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225]),
            transforms.RandomErasing(p=aug["random_erasing_prob"]),
        ])
    else:
        transform = transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225]),
        ])
    return transform


class BreastCancerDataset(Dataset):
    """Custom Dataset for breast cancer classification."""

    def __init__(self, image_paths, labels, transform=None, sources=None, group_ids=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
        self.sources = sources
        self.group_ids = group_ids

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]

        try:
            image = Image.open(img_path).convert("RGB")
        except Exception as e:
            print(f"  [WARNING] Could not load {img_path}: {e}")
            image = Image.new("RGB", (IMAGE_SIZE, IMAGE_SIZE), (0, 0, 0))

        if self.transform:
            image = self.transform(image)

        return image, label


print("Transforms and Dataset class loaded!")

---
## 6. Create DataLoaders

In [ ]:
# ============================================================
# DATALOADER CREATION
# ============================================================

def _verify_no_group_leakage(train_groups, val_groups, test_groups):
    """Verify that no patient/slide group appears in more than one split."""
    train_set = set(train_groups)
    val_set = set(val_groups)
    test_set = set(test_groups)

    tv_leak = train_set & val_set
    tt_leak = train_set & test_set
    vt_leak = val_set & test_set

    if tv_leak or tt_leak or vt_leak:
        print("  [ERROR] DATA LEAKAGE DETECTED!")
        raise ValueError("Data leakage: patient groups overlap between splits!")
    else:
        print("  [OK] No data leakage: all patient groups are split-exclusive.")


def create_dataloaders():
    """Create train, validation, and test dataloaders with patient-level splitting."""
    image_paths, labels, sources, group_ids = collect_all_image_paths()

    image_paths = np.array(image_paths)
    labels = np.array(labels)
    sources = np.array(sources)
    group_ids = np.array(group_ids)

    indices = np.arange(len(image_paths))

    # Split 1: train+val / test
    gss1 = GroupShuffleSplit(n_splits=1, test_size=TEST_RATIO, random_state=RANDOM_SEED)
    trainval_idx, test_idx = next(gss1.split(indices, labels, groups=group_ids))

    # Split 2: train / val
    val_relative = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)
    gss2 = GroupShuffleSplit(n_splits=1, test_size=val_relative, random_state=RANDOM_SEED)
    train_idx_rel, val_idx_rel = next(gss2.split(
        trainval_idx, labels[trainval_idx], groups=group_ids[trainval_idx]
    ))
    train_idx = trainval_idx[train_idx_rel]
    val_idx = trainval_idx[val_idx_rel]

    X_train, y_train = image_paths[train_idx], labels[train_idx]
    X_val, y_val = image_paths[val_idx], labels[val_idx]
    X_test, y_test = image_paths[test_idx], labels[test_idx]
    s_train, s_val, s_test = sources[train_idx], sources[val_idx], sources[test_idx]
    g_train, g_val, g_test = group_ids[train_idx], group_ids[val_idx], group_ids[test_idx]

    # Verify no leakage
    _verify_no_group_leakage(g_train, g_val, g_test)

    print(f"\nSplit sizes (patient-level, leak-free):")
    print(f"  Train: {len(X_train)} (Benign: {(y_train==0).sum()}, Malignant: {(y_train==1).sum()})")
    print(f"  Val:   {len(X_val)}  (Benign: {(y_val==0).sum()}, Malignant: {(y_val==1).sum()})")
    print(f"  Test:  {len(X_test)}  (Benign: {(y_test==0).sum()}, Malignant: {(y_test==1).sum()})")

    # Transforms
    train_transform = get_transforms(is_training=True)
    eval_transform = get_transforms(is_training=False)

    # Datasets
    train_dataset = BreastCancerDataset(X_train.tolist(), y_train.tolist(), train_transform, group_ids=g_train.tolist())
    val_dataset = BreastCancerDataset(X_val.tolist(), y_val.tolist(), eval_transform, group_ids=g_val.tolist())
    test_dataset = BreastCancerDataset(X_test.tolist(), y_test.tolist(), eval_transform, sources=s_test.tolist(), group_ids=g_test.tolist())

    # Weighted sampler for class imbalance
    class_counts = np.bincount(y_train)
    class_weights = 1.0 / class_counts
    sample_weights = class_weights[y_train]
    sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

    # Dataloaders
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, 
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, 
                            num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, 
                             num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

    return train_loader, val_loader, test_loader


print("DataLoader creation function loaded!")

---
## 7. Training Utilities (Mixup, Scheduler, Early Stopping)

In [ ]:
# ============================================================
# TRAINING UTILITIES
# ============================================================

def mixup_data(x, y, alpha=0.2):
    """Apply mixup augmentation."""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0

    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)

    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    """Compute loss for mixup-augmented inputs."""
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


class WarmupCosineScheduler:
    """Linear warmup + cosine annealing LR scheduler."""

    def __init__(self, optimizer, warmup_epochs, total_epochs, lr_max, lr_min=1e-6):
        self.optimizer = optimizer
        self.warmup_epochs = warmup_epochs
        self.total_epochs = total_epochs
        self.lr_max = lr_max
        self.lr_min = lr_min
        self.current_epoch = 0

    def step(self):
        self.current_epoch += 1
        lr = self._compute_lr(self.current_epoch)
        for param_group in self.optimizer.param_groups:
            param_group["lr"] = lr

    def _compute_lr(self, epoch):
        if epoch <= self.warmup_epochs:
            return self.lr_min + (self.lr_max - self.lr_min) * (epoch / self.warmup_epochs)
        else:
            progress = (epoch - self.warmup_epochs) / (self.total_epochs - self.warmup_epochs)
            return self.lr_min + 0.5 * (self.lr_max - self.lr_min) * (1 + math.cos(math.pi * progress))


class EarlyStopping:
    """Early stopping based on validation loss."""

    def __init__(self, patience=7, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_val_loss = None
        self.early_stop = False

    def __call__(self, val_loss, train_loss=None):
        if self.best_val_loss is None:
            self.best_val_loss = val_loss
            return False

        if val_loss < self.best_val_loss - self.min_delta:
            self.best_val_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
                return True
        return False


def freeze_backbone(model):
    """Freeze all parameters except the classifier head."""
    head_keywords = {"head", "classifier", "fc", "last_linear"}
    head_module_names = set()
    for attr_name in dir(model):
        if attr_name in head_keywords:
            attr = getattr(model, attr_name, None)
            if isinstance(attr, nn.Module):
                head_module_names.add(attr_name)

    if not head_module_names:
        children = list(model.named_children())
        if children:
            head_module_names.add(children[-1][0])

    frozen_count = 0
    trainable_count = 0
    for name, param in model.named_parameters():
        is_head = any(name.startswith(head_name + ".") or name == head_name for head_name in head_module_names)
        if is_head:
            param.requires_grad = True
            trainable_count += 1
        else:
            param.requires_grad = False
            frozen_count += 1

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"  Backbone FROZEN: {frozen_count} param groups, Trainable: {trainable:,} / {total:,}")


def unfreeze_all(model):
    """Unfreeze all parameters for full fine-tuning."""
    for param in model.parameters():
        param.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  All layers UNFROZEN: {trainable:,} trainable parameters")


print("Training utilities loaded!")

---
## 8. Training and Evaluation Functions

In [ ]:
# ============================================================
# TRAINING AND EVALUATION FUNCTIONS
# ============================================================

def train_one_epoch(model, dataloader, criterion, optimizer, device, scaler=None, use_mixup=True, mixup_alpha=0.2):
    """Train the model for one epoch."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(dataloader, desc="  Training", leave=False)
    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if use_mixup and mixup_alpha > 0:
            images, labels_a, labels_b, lam = mixup_data(images, labels, mixup_alpha)
        else:
            labels_a = labels
            labels_b = labels
            lam = 1.0

        optimizer.zero_grad(set_to_none=True)

        if scaler is not None:
            with torch.amp.autocast("cuda"):
                outputs = model(images)
                loss = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_MAX_NORM)
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_MAX_NORM)
            optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels_a).sum().item()
        total += labels_a.size(0)

        pbar.set_postfix(loss=f"{loss.item():.4f}")

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    """Evaluate the model on validation/test data."""
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    all_probs = []

    for images, labels in tqdm(dataloader, desc="  Evaluating", leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if device.type == "cuda":
            with torch.amp.autocast("cuda"):
                outputs = model(images)
                loss = criterion(outputs, labels)
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        probs = torch.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    return epoch_loss, epoch_acc, np.array(all_preds), np.array(all_labels), np.array(all_probs)


def compute_all_metrics(y_true, y_pred, y_probs):
    """Compute comprehensive classification metrics."""
    y_prob_positive = y_probs[:, 1]

    try:
        roc_auc = float(roc_auc_score(y_true, y_prob_positive))
    except ValueError:
        roc_auc = 0.0

    try:
        avg_precision = float(average_precision_score(y_true, y_prob_positive))
    except ValueError:
        avg_precision = 0.0

    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="binary", zero_division=0)),
        "recall_sensitivity": float(recall_score(y_true, y_pred, average="binary", zero_division=0)),
        "specificity": float(recall_score(y_true, y_pred, average="binary", pos_label=0, zero_division=0)),
        "f1_score": float(f1_score(y_true, y_pred, average="binary", zero_division=0)),
        "roc_auc": roc_auc,
        "average_precision": avg_precision,
        "matthews_corrcoef": float(matthews_corrcoef(y_true, y_pred)),
        "cohen_kappa": float(cohen_kappa_score(y_true, y_pred)),
    }

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    metrics["true_positives"] = int(tp)
    metrics["true_negatives"] = int(tn)
    metrics["false_positives"] = int(fp)
    metrics["false_negatives"] = int(fn)

    return metrics


print("Training and evaluation functions loaded!")

---
## 9. Full Training Loop

In [ ]:
# ============================================================
# FULL TRAINING LOOP
# ============================================================

def train_model(model, train_loader, val_loader, model_name, device):
    """Full training loop with all anti-overfit measures."""
    print(f"\n{'='*60}")
    print(f"  Training: {model_name}")
    print(f"  Epochs: {NUM_EPOCHS} | Batch: {BATCH_SIZE}")
    print(f"  LR: {LEARNING_RATE} | Weight Decay: {WEIGHT_DECAY}")
    print(f"{'='*60}")

    model = model.to(device)

    # Phase 1: Freeze backbone
    print(f"\n--- Phase 1: Training classifier head only ---")
    freeze_backbone(model)

    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = WarmupCosineScheduler(optimizer, WARMUP_EPOCHS, NUM_EPOCHS, LEARNING_RATE, LR_MIN)
    scaler = torch.amp.GradScaler("cuda") if device.type == "cuda" else None
    early_stopping = EarlyStopping(EARLY_STOPPING_PATIENCE, EARLY_STOPPING_MIN_DELTA)

    history = {
        "train_loss": [], "train_acc": [],
        "val_loss": [], "val_acc": [],
        "lr": [],
    }

    best_val_loss = float("inf")
    best_val_acc = 0.0
    best_model_wts = copy.deepcopy(model.state_dict())
    best_epoch = 0
    backbone_unfrozen = False

    start_time = time.time()

    for epoch in range(1, NUM_EPOCHS + 1):
        epoch_start = time.time()
        print(f"\nEpoch {epoch}/{NUM_EPOCHS}")

        # Phase 2: Unfreeze after FREEZE_BACKBONE_EPOCHS
        if epoch == FREEZE_BACKBONE_EPOCHS + 1 and not backbone_unfrozen:
            print(f"\n--- Phase 2: Unfreezing entire model ---")
            unfreeze_all(model)
            backbone_unfrozen = True

            # Recreate optimizer with differential learning rates
            head_keywords = {"head", "classifier", "fc", "last_linear"}
            head_module_names = set()
            for attr_name in dir(model):
                if attr_name in head_keywords:
                    attr = getattr(model, attr_name, None)
                    if isinstance(attr, nn.Module):
                        head_module_names.add(attr_name)

            backbone_params = []
            head_params = []
            for name, param in model.named_parameters():
                is_head = any(name.startswith(h + ".") or name == h for h in head_module_names)
                if is_head:
                    head_params.append(param)
                else:
                    backbone_params.append(param)

            optimizer = torch.optim.AdamW([
                {"params": backbone_params, "lr": LEARNING_RATE * 0.1},
                {"params": head_params, "lr": LEARNING_RATE},
            ], weight_decay=WEIGHT_DECAY)

            scheduler = WarmupCosineScheduler(optimizer, 1, NUM_EPOCHS - epoch + 1, LEARNING_RATE * 0.1, LR_MIN)

        # Train
        use_mixup = backbone_unfrozen and MIXUP_ALPHA > 0
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device, scaler, use_mixup, MIXUP_ALPHA)

        # Validate
        val_loss, val_acc, _, _, _ = evaluate(model, val_loader, criterion, device)

        current_lr = optimizer.param_groups[0]["lr"]
        scheduler.step()

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["lr"].append(current_lr)

        print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"  Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")
        print(f"  LR: {current_lr:.2e} | Time: {time.time() - epoch_start:.1f}s")

        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            ckpt_path = os.path.join(CHECKPOINT_DIR, f"{model_name}_best.pth")
            torch.save({"epoch": best_epoch, "model_state_dict": best_model_wts, "val_loss": best_val_loss}, ckpt_path)
            print(f"  >> Saved best model (epoch {best_epoch})")

        # Early stopping
        if backbone_unfrozen and early_stopping(val_loss, train_loss):
            print(f"\n  Early stopping at epoch {epoch}")
            break

    elapsed = time.time() - start_time
    print(f"\nTraining completed in {elapsed/60:.1f} minutes")
    print(f"Best val_acc: {best_val_acc:.4f} at epoch {best_epoch}")

    model.load_state_dict(best_model_wts)
    history["training_time_seconds"] = elapsed
    history["best_epoch"] = best_epoch

    return model, history


print("Full training loop loaded!")

---
## 10. Model Definitions

In [ ]:
# ============================================================
# MODEL DEFINITIONS
# ============================================================

class BaseClassifier(nn.Module):
    """Base classifier wrapper with custom head."""
    
    def __init__(self, backbone, in_features, num_classes, dropout=0.3):
        super().__init__()
        self.backbone = backbone
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.LayerNorm(in_features),
            nn.Dropout(p=dropout),
            nn.Linear(in_features, 256),
            nn.GELU(),
            nn.Dropout(p=dropout * 0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.backbone.forward_features(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x


def build_model(model_type, num_classes=NUM_CLASSES, pretrained=True, dropout=DROPOUT_RATE):
    """Build a model by type (maxvit, mobilevit, efficientvit)."""
    model_cfg = MODELS[model_type]
    model_name = model_cfg["name"]

    print(f"\nBuilding model: {model_name}")
    print(f"  Description: {model_cfg['description']}")

    backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0)

    with torch.no_grad():
        dummy = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE)
        features = backbone.forward_features(dummy)
        if features.dim() == 4:
            in_features = features.shape[1]
        else:
            in_features = features.shape[-1]

    print(f"  Feature dim: {in_features}")

    model = BaseClassifier(backbone, in_features, num_classes, dropout)

    total_params = sum(p.numel() for p in model.parameters())
    print(f"  Total parameters: {total_params:,}")

    return model


class HybridViTClassifier(nn.Module):
    """Feature-level fusion of MaxViT + MobileViT."""

    def __init__(self, backbone_a, backbone_b, feat_dim_a, feat_dim_b, num_classes, dropout=0.3):
        super().__init__()
        self.backbone_a = backbone_a
        self.backbone_b = backbone_b
        self.pool = nn.AdaptiveAvgPool2d(1)

        fused_dim = feat_dim_a + feat_dim_b

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.LayerNorm(fused_dim),
            nn.Dropout(p=dropout),
            nn.Linear(fused_dim, 512),
            nn.GELU(),
            nn.Dropout(p=dropout * 0.5),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(p=dropout * 0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        feat_a = self.backbone_a.forward_features(x)
        feat_b = self.backbone_b.forward_features(x)
        feat_a = self.pool(feat_a)
        feat_b = self.pool(feat_b)
        fused = torch.cat([feat_a, feat_b], dim=1)
        out = self.classifier(fused)
        return out


def build_hybrid_model(num_classes=NUM_CLASSES, pretrained=True, dropout=DROPOUT_RATE):
    """Build HybridViT model: MaxViT + MobileViT feature fusion."""
    cfg_a = MODELS["maxvit"]
    cfg_b = MODELS["mobilevit"]

    print(f"\nBuilding HybridViT model (feature fusion)")
    print(f"  Backbone A: {cfg_a['name']}")
    print(f"  Backbone B: {cfg_b['name']}")

    backbone_a = timm.create_model(cfg_a["name"], pretrained=pretrained, num_classes=0)
    backbone_b = timm.create_model(cfg_b["name"], pretrained=pretrained, num_classes=0)

    with torch.no_grad():
        dummy = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE)
        feat_a = backbone_a.forward_features(dummy)
        feat_b = backbone_b.forward_features(dummy)
        feat_dim_a = feat_a.shape[1] if feat_a.dim() == 4 else feat_a.shape[-1]
        feat_dim_b = feat_b.shape[1] if feat_b.dim() == 4 else feat_b.shape[-1]

    print(f"  Feature dim A: {feat_dim_a}")
    print(f"  Feature dim B: {feat_dim_b}")
    print(f"  Fused dim: {feat_dim_a + feat_dim_b}")

    model = HybridViTClassifier(backbone_a, backbone_b, feat_dim_a, feat_dim_b, num_classes, dropout)

    total_params = sum(p.numel() for p in model.parameters())
    print(f"  Total parameters: {total_params:,}")

    return model


print("Model definitions loaded!")

---
## 11. Visualization Functions

In [ ]:
# ============================================================
# VISUALIZATION FUNCTIONS
# ============================================================

def plot_training_curves(history, model_name, save_dir):
    """Plot training curves."""
    os.makedirs(save_dir, exist_ok=True)
    epochs = range(1, len(history["train_loss"]) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(epochs, history["train_loss"], "b-o", label="Train Loss", markersize=4)
    axes[0].plot(epochs, history["val_loss"], "r-o", label="Val Loss", markersize=4)
    axes[0].set_title(f"{model_name} - Loss Curves")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs, history["train_acc"], "b-o", label="Train Acc", markersize=4)
    axes[1].plot(epochs, history["val_acc"], "r-o", label="Val Acc", markersize=4)
    axes[1].set_title(f"{model_name} - Accuracy Curves")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"{model_name}_training_curves.png"), dpi=150)
    plt.show()
    plt.close()


def plot_confusion_matrix(y_true, y_pred, model_name, save_dir):
    """Plot confusion matrix."""
    os.makedirs(save_dir, exist_ok=True)
    cm = confusion_matrix(y_true, y_pred)

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title(f"{model_name} - Confusion Matrix")
    ax.set_ylabel("True Label")
    ax.set_xlabel("Predicted Label")

    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"{model_name}_confusion_matrix.png"), dpi=150)
    plt.show()
    plt.close()


def plot_roc_curve(y_true, y_probs, model_name, save_dir):
    """Plot ROC curve."""
    os.makedirs(save_dir, exist_ok=True)
    y_prob_pos = y_probs[:, 1]
    fpr, tpr, _ = roc_curve(y_true, y_prob_pos)
    auc_score = roc_auc_score(y_true, y_prob_pos)

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(fpr, tpr, "b-", linewidth=2, label=f"ROC Curve (AUC = {auc_score:.4f})")
    ax.plot([0, 1], [0, 1], "r--", linewidth=1, label="Random")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"{model_name} - ROC Curve")
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"{model_name}_roc_curve.png"), dpi=150)
    plt.show()
    plt.close()


print("Visualization functions loaded!")

---
## 12. Run Training

### Select which model to train by uncommenting the appropriate section

In [ ]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Load data
print("\n--- Loading Data ---")
train_loader, val_loader, test_loader = create_dataloaders()

### Train MaxViT

In [ ]:
# Train MaxViT
print("\n--- Building MaxViT Model ---")
maxvit_model = build_model("maxvit")

maxvit_model, maxvit_history = train_model(maxvit_model, train_loader, val_loader, "MaxViT", device)

# Evaluate
print("\n--- Evaluating MaxViT ---")
criterion = nn.CrossEntropyLoss()
_, _, preds, labels, probs = evaluate(maxvit_model, test_loader, criterion, device)
maxvit_metrics = compute_all_metrics(labels, preds, probs)

print("\nMaxViT Results:")
for k, v in maxvit_metrics.items():
    print(f"  {k}: {v}")

# Plot
save_dir = os.path.join(RESULTS_DIR, "MaxViT")
plot_training_curves(maxvit_history, "MaxViT", save_dir)
plot_confusion_matrix(labels, preds, "MaxViT", save_dir)
plot_roc_curve(labels, probs, "MaxViT", save_dir)

### Train MobileViT

In [ ]:
# Train MobileViT
print("\n--- Building MobileViT Model ---")
mobilevit_model = build_model("mobilevit")

mobilevit_model, mobilevit_history = train_model(mobilevit_model, train_loader, val_loader, "MobileViT", device)

# Evaluate
print("\n--- Evaluating MobileViT ---")
_, _, preds, labels, probs = evaluate(mobilevit_model, test_loader, criterion, device)
mobilevit_metrics = compute_all_metrics(labels, preds, probs)

print("\nMobileViT Results:")
for k, v in mobilevit_metrics.items():
    print(f"  {k}: {v}")

# Plot
save_dir = os.path.join(RESULTS_DIR, "MobileViT")
plot_training_curves(mobilevit_history, "MobileViT", save_dir)
plot_confusion_matrix(labels, preds, "MobileViT", save_dir)
plot_roc_curve(labels, probs, "MobileViT", save_dir)

### Train EfficientViT

In [ ]:
# Train EfficientViT
print("\n--- Building EfficientViT Model ---")
efficientvit_model = build_model("efficientvit")

efficientvit_model, efficientvit_history = train_model(efficientvit_model, train_loader, val_loader, "EfficientViT", device)

# Evaluate
print("\n--- Evaluating EfficientViT ---")
_, _, preds, labels, probs = evaluate(efficientvit_model, test_loader, criterion, device)
efficientvit_metrics = compute_all_metrics(labels, preds, probs)

print("\nEfficientViT Results:")
for k, v in efficientvit_metrics.items():
    print(f"  {k}: {v}")

# Plot
save_dir = os.path.join(RESULTS_DIR, "EfficientViT")
plot_training_curves(efficientvit_history, "EfficientViT", save_dir)
plot_confusion_matrix(labels, preds, "EfficientViT", save_dir)
plot_roc_curve(labels, probs, "EfficientViT", save_dir)

### Train HybridViT (Novel Architecture)

In [ ]:
# Train HybridViT (Novel Feature Fusion)
print("\n--- Building HybridViT Model ---")
hybrid_model = build_hybrid_model()

hybrid_model, hybrid_history = train_model(hybrid_model, train_loader, val_loader, "HybridViT", device)

# Evaluate
print("\n--- Evaluating HybridViT ---")
_, _, preds, labels, probs = evaluate(hybrid_model, test_loader, criterion, device)
hybrid_metrics = compute_all_metrics(labels, preds, probs)

print("\nHybridViT Results:")
for k, v in hybrid_metrics.items():
    print(f"  {k}: {v}")

# Plot
save_dir = os.path.join(RESULTS_DIR, "HybridViT")
plot_training_curves(hybrid_history, "HybridViT", save_dir)
plot_confusion_matrix(labels, preds, "HybridViT", save_dir)
plot_roc_curve(labels, probs, "HybridViT", save_dir)

---
## 13. Compare All Models

In [ ]:
# Compare all models (run after training all)
import pandas as pd

# Collect results (make sure all models are trained)
all_results = {}

try:
    all_results["MaxViT"] = maxvit_metrics
except:
    pass

try:
    all_results["MobileViT"] = mobilevit_metrics
except:
    pass

try:
    all_results["EfficientViT"] = efficientvit_metrics
except:
    pass

try:
    all_results["HybridViT"] = hybrid_metrics
except:
    pass

if all_results:
    df = pd.DataFrame(all_results).T
    print("\n" + "="*60)
    print("MODEL COMPARISON")
    print("="*60)
    print(df[["accuracy", "precision", "recall_sensitivity", "specificity", "f1_score", "roc_auc", "matthews_corrcoef"]].to_string())
else:
    print("No models trained yet. Run the training cells above first.")

---
## 14. Save Results

In [ ]:
# Save all results to JSON
import json

if all_results:
    results_path = os.path.join(RESULTS_DIR, "all_models_comparison.json")
    with open(results_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"Results saved to: {results_path}")

---
## Done!

This notebook contains the complete training pipeline for breast cancer classification using Vision Transformers.

**Models trained:**
- MaxViT (Multi-Axis Vision Transformer)
- MobileViT (Lightweight Mobile Vision Transformer)
- EfficientViT (Optimized Efficient Vision Transformer)
- HybridViT (Novel Feature-Level Fusion)

**Results are saved in:** `results/` directory